In [ ]:
project_path = "/home/jupyter"
import os
import sys

sys.path.append(project_path)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from google.cloud import bigquery

from fintrans_toolbox.src import bq_utils as bq
from fintrans_toolbox.src import table_utils as t


client = bigquery.Client()


In [ ]:
# Summarise UK Cardholders Spending Online Total --------------------------------------------

UK_spending_Online = '''SELECT 
time_period_value, SUM(spend) AS total_spend

FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` 
WHERE time_period_value IN (
    '2019Q1', '2019Q2', '2019Q3', '2019Q4', '2020Q1', '2020Q2', '2020Q3', '2020Q4',
    '2021Q1', '2021Q2', '2021Q3', '2021Q4', '2022Q1', '2022Q2', '2022Q3', '2022Q4',
    '2023Q1', '2023Q2', '2023Q3', '2023Q4', '2024Q1', '2024Q2', '2024Q3', '2024Q4', '2025Q1'
  )
and mcg = 'All' 
and merchant_channel = 'Online' 
and cardholder_origin_country = 'All' 
and cardholder_origin = 'UNITED KINGDOM' 
GROUP BY 
time_period_value, total_spend 
ORDER BY time_period_value DESC'''
df_by_Online_All = bq.read_bq_table_sql(client, UK_spending_Online)
df_by_Online_All.head()

# Summarise the data by country -----------------------------------------------------------------
UK_spending_by_country00 = '''SELECT time_period_value, destination_country, spend FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` where time_period = 'Quarter' and mcg = 'All' and merchant_channel = 'All' and cardholder_origin_country = 'All' and cardholder_origin = 'UNITED KINGDOM' and destination_country = 'UNITED KINGDOM' GROUP BY destination_country, 
time_period_value, spend ORDER BY time_period_value, destination_country DESC'''
df_by_country00 = bq.read_bq_table_sql(client, UK_spending_by_country00)
# Rename the 'spend' column to 'online_spend'
df_by_country00 = df_by_country00.rename(columns={'spend': 'total_spend'})
df_by_country00.head()